## Setup

In [72]:
import os

os.environ["KERAS_BACKEND"] = "tensorflow"

import keras
from keras import layers
from keras import ops
from keras.layers import TextVectorization
import numpy as np
import os
import string
import random
import tensorflow
import tensorflow.data as tf_data
import tensorflow.strings as tf_strings

## Implement a Transformer block as a layer

In [73]:
def causal_attention_mask(batch_size, n_dest, n_src, dtype):
    """
    Mask the upper half of the dot product matrix in self attention.
    This prevents flow of information from future tokens to current token.
    1's in the lower triangle, counting from the lower right corner.
    """
    i = ops.arange(n_dest)[:, None]
    j = ops.arange(n_src)
    m = i >= j - n_src + n_dest
    mask = ops.cast(m, dtype)
    mask = ops.reshape(mask, [1, n_dest, n_src])
    # Use a direct list for multiples argument of ops.tile to avoid OperatorNotAllowedInGraphError
    # when batch_size is a symbolic tensor.
    multiples = [batch_size, 1, 1]
    return ops.tile(mask, multiples)


class TransformerBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1):
        super().__init__()
        self.att = layers.MultiHeadAttention(num_heads, embed_dim)
        self.ffn = keras.Sequential(
            [
                layers.Dense(ff_dim, activation="relu"),
                layers.Dense(embed_dim),
            ]
        )
        self.layernorm1 = layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = layers.Dropout(rate)
        self.dropout2 = layers.Dropout(rate)

    def call(self, inputs):
        input_shape = ops.shape(inputs)
        batch_size = input_shape[0]
        seq_len = input_shape[1]
        causal_mask = causal_attention_mask(batch_size, seq_len, seq_len, "bool")
        attention_output = self.att(inputs, inputs, attention_mask=causal_mask)
        attention_output = self.dropout1(attention_output)
        out1 = self.layernorm1(inputs + attention_output)
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output)
        return self.layernorm2(out1 + ffn_output)

## Implement an embedding layer

Create two separate embedding layers: one for tokens and one for token index
(positions).

In [74]:
class TokenAndPositionEmbedding(layers.Layer):
    def __init__(self, maxlen, vocab_size, embed_dim):
        super().__init__()
        self.token_emb = layers.Embedding(input_dim=vocab_size, output_dim=embed_dim)
        self.pos_emb = layers.Embedding(input_dim=maxlen, output_dim=embed_dim)

    def call(self, x):
        maxlen = ops.shape(x)[-1]
        positions = ops.arange(0, maxlen, 1)
        positions = self.pos_emb(positions)
        x = self.token_emb(x)
        return x + positions

## Implement the miniature GPT model

In [75]:
vocab_size = 20000  # Only consider the top 20k words
maxlen = 100  # Max sequence size
embed_dim = 256  # Embedding size for each token
num_heads = 4  # Number of attention heads
feed_forward_dim = 256  # Hidden layer size in feed forward network inside transformer


def create_model():
    inputs = layers.Input(shape=(maxlen,), dtype="int32")
    embedding_layer = TokenAndPositionEmbedding(maxlen, vocab_size, embed_dim)
    x = embedding_layer(inputs)
    transformer_block = TransformerBlock(embed_dim, num_heads, feed_forward_dim)
    x = transformer_block(x)
    outputs = layers.Dense(vocab_size)(x)
    model = keras.Model(inputs=inputs, outputs=[outputs, x])
    loss_fn = keras.losses.SparseCategoricalCrossentropy(from_logits=True)
    model.compile(
        "adam",
        loss=[loss_fn, None],
    )  # No loss and optimization based on word embeddings from transformer block
    return model

## Prepare the data for word-level language modelling

Download the IMDB dataset and combine training and validation sets for a text
generation task.

In [76]:
filenames = ["simpsons-transcripts.txt"]

In [77]:
import keras
from keras import layers
from keras.layers import TextVectorization
import numpy as np
import os
import string
import random
import tensorflow
import tensorflow.data as tf_data
import tensorflow.strings as tf_strings

vocab_size = 20000  # Only consider the top 20k words
maxlen = 80  # Max sequence size

batch_size = 128

# Create a dataset from the text file
text_ds = tf_data.TextLineDataset(filenames)
text_ds = text_ds.shuffle(buffer_size=256)
text_ds = text_ds.batch(batch_size)

def custom_standardization(input_string):
    """Remove all punctuation except periods"""
    lowercased = tf_strings.lower(input_string)
    # Remove double quotation marks (already handled if we remove all punctuation, but kept for clarity)
    no_quotes = tf_strings.regex_replace(lowercased, '"', '')

    # Get all punctuation except the period
    punctuation_to_remove = string.punctuation.replace('.', '')
    # Use a regex to remove these punctuation marks
    # We need to escape special regex characters in the punctuation string
    return tf_strings.regex_replace(no_quotes, f"([{re.escape(punctuation_to_remove)}])", '')


# Create a vectorization layer and adapt it to the text
vectorize_layer = TextVectorization(
    standardize=custom_standardization,
    max_tokens=vocab_size - 1,
    output_mode="int",
    output_sequence_length=maxlen + 1,
)
vectorize_layer.adapt(text_ds)
vocab = vectorize_layer.get_vocabulary()  # To get words back from token indices

def prepare_lm_inputs_labels(text):
    """
    Shift word sequences by 1 position so that the target for position (i) is
    word at position (i+1). The model will use all words up till position (i)
    to predict the next word.
    """
    text = tensorflow.expand_dims(text, -1)
    tokenized_sentences = vectorize_layer(text)
    x = tokenized_sentences[:, :-1]
    y = tokenized_sentences[:, 1:]
    return x, y


text_ds = text_ds.map(prepare_lm_inputs_labels, num_parallel_calls=tf_data.AUTOTUNE)
text_ds = text_ds.prefetch(tf_data.AUTOTUNE)

## Implement a Keras callback for generating text

In [78]:
import re

class TextGenerator(keras.callbacks.Callback):
    """A callback to generate text from a trained model.
    1. Feed some starting prompt to the model
    2. Predict probabilities for the next token
    3. Sample the next token and add it to the next input

    Arguments:
        max_tokens: Integer, the number of tokens to be generated after prompt.
        start_tokens: List of integers, the token indices for the starting prompt.
        index_to_word: List of strings, obtained from the TextVectorization layer.
        top_k: Integer, sample from the `top_k` token predictions.
        print_every: Integer, print after this many epochs.
        temperature: Float, controls randomness of predictions.
    """

    def __init__(
        self, max_tokens, start_tokens, index_to_word, top_k=10, print_every=1, temperature=1.0
    ):
        self.max_tokens = max_tokens
        self.start_tokens = start_tokens
        self.index_to_word = index_to_word
        self.print_every = print_every
        self.k = top_k
        self.temperature = temperature

    def sample_from(self, logits):
        logits, indices = ops.top_k(logits, k=self.k, sorted=True)
        indices = np.asarray(indices).astype("int32")
        # Apply temperature to logits
        logits = logits / self.temperature
        preds = keras.activations.softmax(ops.expand_dims(logits, 0))[0]
        preds = np.asarray(preds).astype("float32")
        return np.random.choice(indices, p=preds)

    def detokenize(self, number):
        return self.index_to_word[number]

    def on_epoch_end(self, epoch, logs=None):
        start_tokens = [_ for _ in self.start_tokens]
        if (epoch + 1) % self.print_every != 0:
            return
        num_tokens_generated = 0
        tokens_generated = []
        while num_tokens_generated <= self.max_tokens:
            pad_len = maxlen - len(start_tokens)
            sample_index = len(start_tokens) - 1
            if pad_len < 0:
                x = start_tokens[:maxlen]
                sample_index = maxlen - 1
            elif pad_len > 0:
                x = start_tokens + [0] * pad_len
            else:
                x = start_tokens
            x = np.array([x])
            y, _ = self.model.predict(x, verbose=0)
            sample_token = self.sample_from(y[0][sample_index])
            tokens_generated.append(sample_token)
            start_tokens.append(sample_token)
            num_tokens_generated = len(tokens_generated)

        # Combine start tokens and generated tokens for cleaning
        all_tokens = self.start_tokens + tokens_generated

        # Filter out unwanted tokens and detokenize
        cleaned_words = []
        for token_idx in all_tokens:
            word = self.detokenize(token_idx)
            # Filter out [UNK], '^', and tokens that are just digits (like "02", "15")
            if word == "[UNK]" or word == "^" or (word.isdigit() and len(word) <= 2):
                continue
            cleaned_words.append(word)

        txt = " ".join(cleaned_words)

        # Further clean up spacing and punctuation
        # Remove space before period: "word ." -> "word."
        txt = re.sub(r'\s(\.)', r'\1', txt)
        # Ensure space after period: "word.next" -> "word. next"
        txt = re.sub(r'(\.)(\S)', r'\1 \2', txt)
        # Remove multiple spaces and strip leading/trailing spaces
        txt = re.sub(r'\s+', ' ', txt).strip()

        # Capitalize the first letter of the generated text
        if txt:
            txt = txt[0].upper() + txt[1:]

        # Ensure it ends with a period if not already ending with a period
        if txt and not txt.endswith("."):
            txt += "."

        print(f"generated text:\n{txt}\n")


# Tokenize starting prompt
word_to_index = {}
for index, word in enumerate(vocab):
    word_to_index[word] = index

start_prompt = "Springfield has been surrounded by"
start_tokens = [word_to_index.get(_, 1) for _ in start_prompt.split()]
num_tokens_generated = 25

text_gen_callback = TextGenerator(num_tokens_generated, start_tokens, vocab, temperature=1.25)

## Train the model

Note: This code should preferably be run on GPU.

In [79]:
model = create_model()

model.fit(text_ds, verbose=2, epochs=50, callbacks=[text_gen_callback])

Epoch 1/50


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


generated text:
Has been surrounded by donuts but and how with an with to but she and you and but a with a and a a the a and the.

6/6 - 19s - 3s/step - loss: 9.4188
Epoch 2/50
generated text:
Has been surrounded by donuts to your in the i for in a but but i to you but the but to to you i to and to but a.

6/6 - 3s - 499ms/step - loss: 7.8830
Epoch 3/50
generated text:
Has been surrounded by donuts i i the you a the a the to i and in this the a i you the in for the i to.

6/6 - 3s - 497ms/step - loss: 7.0424
Epoch 4/50
generated text:
Has been surrounded by donuts in a a the i to the the of of to and i the and you the in and to a.

6/6 - 4s - 616ms/step - loss: 6.7704
Epoch 5/50
generated text:
Has been surrounded by donuts i a and to simpsons to and the a doh i the you i the doh to the.

6/6 - 4s - 747ms/step - loss: 6.7413
Epoch 6/50
generated text:
Has been surrounded by donuts i the to a you i and the the i in a in the to i.

6/6 - 3s - 504ms/step - loss: 6.7193
Epoch 7/50
generate